# Summary
1. We process FIRMS data to then calculate how many times a pixel burned in the year
2. We then export the data for 2022 to Google Cloud Storage.

We created 14-day periods and checked if an area burned during that time. Then we checked for how many of the 14-day periods a pixel burned during the year. Thus, the maximum number is 26 times a pixel could've burned during the year.

### Future
- Export for all years

# Setting up the environment

In [ ]:
# Import and/or install libraries
import subprocess, os, gcsfs, json

try:
    import geemap, ee
except ImportError:
    subprocess.check_call(["python", '-m', 'pip', 'install', '-U', 'geemap'])
    import geemap, ee

from datetime import datetime
import pandas as pd

In [ ]:
# Connect to Google Drive to access files

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Connect to Google Earth Engine if neccessary

service_account = os.environ.get('GOOGLE_SERVICE_ACCOUNT')
credentials = ee.ServiceAccountCredentials(service_account, os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'))
ee.Initialize(credentials)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
# block_list = list(range(1, 26))

# blocks = []
# with open("/content/drive/MyDrive/data/blocks.geojson") as f:
#     json_data = json.load(f)
#     for block_id in block_list:
#         for feature in json_data['features']:
#             if feature['properties']['id'] == block_id:
#                 feature['properties']['style'] = {}
#                 blocks.append(feature)

# ee_blocks = [ee.Feature(block) for block in blocks]

In [ ]:
# Or import polygon

with open("/content/drive/MyDrive/data/Eastern CAR Wilderness.geojson") as f:
    json_data = json.load(f)

aoi = ee.FeatureCollection(json_data).first()

{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'MultiPolygon', 'coordinates': [[[[23.325506871000073, 5.496208443000057], [23.32551531300004, 5.496209420000071], [23.32547292600003, 5.496193662000053], [23.325506871000073, 5.496208443000057]]], [[[23.343174244000068, 5.533093223000037], [23.343048699000065, 5.533106192000048], [23.343218267000054, 5.533106439000051], [23.343174244000068, 5.533093223000037]]], [[[24.687184595000076, 5.580853753000042], [24.687047343000074, 5.58092126400004], [24.687252869000076, 5.580921939000064], [24.687184595000076, 5.580853753000042]]], [[[23.332437524000056, 5.555920468000068], [23.332005768000045, 5.556116419000034], [23.332006282000066, 5.556546379000054], [23.331712164000066, 5.556898513000021], [23.33194830900004, 5.557328192000057], [23.332007778000047, 5.557797167000047], [23.33216549800005, 5.558324656000025], [23.33183217900006, 5.558735467000076], [23.33151887100007, 5.559458951000067], [23.331697449000046, 5.561002693000034],

In [ ]:
# Specify dates
# year = 2024
# start_date = '{}-01-01'.format(year)
# end_date = '{}-12-31'.format(year)


year = 2025
end_year = year + 1
start_date = '{}-09-01'.format(year)
end_date = '{}-08-18'.format(end_year)

In [ ]:
# Set projection
project_crs = 'EPSG:3857'
project_projection = ee.Projection(project_crs)

# Import firms
img_date_range = [d.strftime('%Y-%m-%d') for d in pd.date_range(start_date, end_date, freq='14d')]
img_date_list = ee.List(img_date_range)


def create_img_for_day(day):
  fire_image = ee.ImageCollection('FIRMS').filter(ee.Filter.date(ee.Date(day), ee.Date(day).advance(14, 'day'))).select('T21').sum().gt(0);
  return fire_image

doy_images = ee.ImageCollection(img_date_list.map(create_img_for_day))

project_proj = doy_images.mean().projection()

fire = doy_images.sum().reproject(project_projection, None, 1000).reduceResolution(reducer = ee.Reducer.sum()).toByte()


In [ ]:
# Loop through blocks and export
# for id, block in enumerate(ee_blocks):

#   image_name = 'fire' + str(year) + '_' + str(id)
#   fileNamePrefix = 'COGS/fire/' + str(year) + '/' + image_name


#   exportConfig = {
#       'image': fire,
#       'description': image_name,
#       'bucket': 'nature-watch-bucket',
#       'fileNamePrefix': fileNamePrefix,
#       'crs': 'EPSG:4326',
#       'scale': 30,
#       'maxPixels': 3984245980,
#       'region': block,
#       'fileFormat': 'GeoTIFF',
#       'formatOptions': {'cloudOptimized': True}
#   }

#   task = ee.batch.Export.image.toCloudStorage(**exportConfig)
#   task.start()

NameError: name 'ee_blocks' is not defined

In [ ]:
# Or just export one block
# id = 13
# block = ee_blocks[id]

# image_name = 'fire' + str(year) + '_' + str(id)
# fileNamePrefix = 'COGS/fire/' + str(year) + '/' + image_name


# exportConfig = {
#     'image': fire,
#     'description': image_name,
#     'bucket': 'nature-watch-bucket',
#     'fileNamePrefix': fileNamePrefix,
#     'crs': 'EPSG:4326',
#     'scale': 30,
#     'maxPixels': 3954245980,
#     'region': block,
#     'fileFormat': 'GeoTIFF',
#     'formatOptions': {'cloudOptimized': True}
# }

# task = ee.batch.Export.image.toCloudStorage(**exportConfig)
# task.start()

In [ ]:
# Or export aoi

image_name = 'fire' + str(year)
fileNamePrefix = 'COGS/temp/chinko/fire/' + image_name


exportConfig = {
    'image': fire,
    'description': image_name,
    'bucket': 'nature-watch-bucket',
    'fileNamePrefix': fileNamePrefix,
    'crs': 'EPSG:4326',
    'scale': 30,
    'maxPixels': 3954245980,
    'region': aoi.geometry(),
    'fileFormat': 'GeoTIFF',
    'formatOptions': {'cloudOptimized': True}
}

task = ee.batch.Export.image.toCloudStorage(**exportConfig)
task.start()


In [ ]:
ee.data.listOperations()

[{'name': 'projects/earthengine-legacy/operations/B4YBMNVYIM5A3EO2GWMZUGIQ',
  'metadata': {'@type': 'type.googleapis.com/google.earthengine.v1alpha.OperationMetadata',
   'state': 'RUNNING',
   'description': 'fire2025',
   'priority': 100,
   'createTime': '2026-09-25T09:39:46.310156Z',
   'updateTime': '2026-09-25T09:39:49.958657Z',
   'startTime': '2026-09-25T09:39:49.896479Z',
   'type': 'EXPORT_IMAGE',
   'attempt': 1}}]

In [ ]:
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.addLayer(fire, {'min':0, 'max':6, 'palette':['yellow', 'orange', 'red'], 'opacity': 1}, 'fire')

Map.setCenter(11.578, 1.600, 12)
Map

Map(center=[1.6, 11.578], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…